# ResNet18 + KAN Hybrid Training

Este notebook implementa una arquitectura h?brida para mamograf?as:

1. Backbone CNN: `ResNet18` preentrenada.
2. Pooling: `Global Average Pooling` para compactar el mapa espacial a un vector latente.
3. Cabezal KAN: `KAN([512, 32, 2], grid=3, k=3)`.
4. Entrenamiento en dos fases:
   - Fase 1: entrenar solo la KAN.
   - Fase 2: descongelar `layer4` y afinar backbone + KAN.
5. Grid updates peri?dicos para adaptar los B-splines al rango real de los embeddings.
6. Etapa opcional final con L-BFGS sobre la KAN.


In [1]:
from pathlib import Path
from collections import Counter
import copy
import importlib.util
import random
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, f1_score, precision_score

warnings.filterwarnings('ignore', message='std\(\): degrees of freedom is <= 0')


<>:21: SyntaxWarning: invalid escape sequence '\('
<>:21: SyntaxWarning: invalid escape sequence '\('
C:\Users\santy\AppData\Local\Temp\ipykernel_7224\2796590260.py:21: SyntaxWarning: invalid escape sequence '\('
  warnings.filterwarnings('ignore', message='std\(\): degrees of freedom is <= 0')


In [ ]:
if not importlib.util.find_spec('kan'):
    raise RuntimeError('KAN no instalado. Ejecuta `%pip install pykan`, reinicia el kernel y corre de nuevo.')

from kan import KAN

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cpu')
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
TRAIN_CSV = ROOT / 'src/data/processed/manifest_train.csv'
TEST_CSV = ROOT / 'src/data/processed/manifest_test.csv'
OUT_DIR = ROOT / 'reports/models'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'resnet18_kan_hybrid_gap32'
BEST_PATH = OUT_DIR / f'{MODEL_NAME}.pt'
HISTORY_PATH = OUT_DIR / f'{MODEL_NAME}_history.csv'

IMG_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 0

PHASE1_EPOCHS = 8
PHASE2_EPOCHS = 10
EARLY_STOP_PATIENCE = 4

PHASE1_LR = 1e-3
PHASE2_LR_KAN = 5e-4
PHASE2_LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4

GRID_UPDATE_ENABLED = True
GRID_UPDATE_AFTER_EPOCH = 1
GRID_UPDATE_EVERY = 1
GRID_UPDATE_MAX_BATCHES = 8

RUN_LBFGS_FINETUNE = True
LBFGS_STEPS = 8
LBFGS_LR = 0.3
LBFGS_MAX_BATCHES = 6

print('DEVICE:', DEVICE)
print('Best path:', BEST_PATH)


DEVICE: cpu
Best path: G:\Cosas_programacion\Breast Cancer Interpretable-ml\reports\models\resnet18_kan_hybrid_gap32.pt


In [3]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

patients = train_df['patient_id'].dropna().unique()
train_pat, val_pat = train_test_split(patients, test_size=0.2, random_state=SEED)

tr_df = train_df[train_df['patient_id'].isin(train_pat)].copy()
val_df = train_df[train_df['patient_id'].isin(val_pat)].copy()

print('Train:', tr_df.shape, tr_df['label_name'].value_counts().to_dict())
print('Val:', val_df.shape, val_df['label_name'].value_counts().to_dict())
print('Test:', test_df.shape, test_df['label_name'].value_counts().to_dict())


Train: (2315, 10) {'BENIGN': 1383, 'MALIGNANT': 932}
Val: (549, 10) {'BENIGN': 300, 'MALIGNANT': 249}
Test: (422, 10) {'BENIGN': 248, 'MALIGNANT': 174}


In [4]:
class MammographyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path_local']).convert('RGB')
        x = self.transform(image)
        y = int(row['label'])
        return x, y


train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.97, 1.03)),
    transforms.ColorJitter(brightness=0.08, contrast=0.12),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(MammographyDataset(tr_df, train_tfms), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
val_loader = DataLoader(MammographyDataset(val_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(MammographyDataset(test_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [5]:
class HybridResNetKAN(nn.Module):
    def __init__(self, kan_hidden=32, kan_grid=3, kan_k=3):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.gap = base.avgpool
        self.kan = KAN(width=[512, kan_hidden, 2], grid=kan_grid, k=kan_k, save_act=True, auto_save=False, seed=SEED)

    def forward_features(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        z = self.gap(x).flatten(1)
        return z

    def forward(self, x):
        z = self.forward_features(x)
        logits = self.kan(z)
        return logits


def freeze_backbone(model):
    for name, param in model.named_parameters():
        if not name.startswith('kan.'):
            param.requires_grad = False
    for param in model.kan.parameters():
        param.requires_grad = True


def unfreeze_layer4_and_kan(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.kan.parameters():
        param.requires_grad = True


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


model = HybridResNetKAN(kan_hidden=32, kan_grid=3, kan_k=3).to(DEVICE)
freeze_backbone(model)
trainable, total = count_trainable_params(model)
print(f'Trainable params: {trainable:,} / {total:,}')
model


Trainable params: 235,848 / 11,412,360


HybridResNetKAN(
  (stem): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (

In [6]:
counts = Counter(tr_df['label'].tolist())
class_weights = torch.tensor([
    len(tr_df) / (2 * counts[0]),
    len(tr_df) / (2 * counts[1]),
], dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
class_weights


tensor([0.8369, 1.2420])

In [7]:
def compute_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)
    return {
        'acc': accuracy_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan'),
        'recall_malignant': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision_malignant': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_malignant': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }


def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_seen = 0
    y_true, y_pred, y_prob = [], [], []

    with torch.set_grad_enabled(training):
        for x, y in loader:
            if x.size(0) < 2:
                continue

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            if training:
                optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            if training:
                loss.backward()
                optimizer.step()

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(1)

            batch_size = y.size(0)
            total_loss += loss.item() * batch_size
            total_seen += batch_size
            y_true.extend(y.detach().cpu().numpy().tolist())
            y_pred.extend(preds.detach().cpu().numpy().tolist())
            y_prob.extend(probs.detach().cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred, y_prob)
    metrics['loss'] = total_loss / total_seen
    return metrics


def maybe_update_kan_grid(loader, max_batches=GRID_UPDATE_MAX_BATCHES):
    if not GRID_UPDATE_ENABLED:
        return False
    if not hasattr(model.kan, 'update_grid_from_samples'):
        return False

    model.eval()
    features = []
    with torch.no_grad():
        for batch_idx, (x, _) in enumerate(loader):
            x = x.to(DEVICE)
            z = model.forward_features(x)
            features.append(z)
            if batch_idx + 1 >= max_batches:
                break

    if not features:
        return False

    samples = torch.cat(features, dim=0)
    model.kan.update_grid_from_samples(samples)
    return True


def build_phase2_optimizer(model):
    return torch.optim.Adam([
        {'params': model.layer4.parameters(), 'lr': PHASE2_LR_BACKBONE},
        {'params': model.kan.parameters(), 'lr': PHASE2_LR_KAN},
    ], weight_decay=WEIGHT_DECAY)


In [8]:
# Smoke test corto de forward/backward
freeze_backbone(model)
smoke_optimizer = torch.optim.Adam(model.kan.parameters(), lr=PHASE1_LR)
model.train()
t0 = time.time()
for i, (x, y) in enumerate(train_loader):
    if i >= 2:
        break
    x = x.to(DEVICE)
    y = y.to(DEVICE)
    smoke_optimizer.zero_grad()
    logits = model(x)
    loss = criterion(logits, y)
    loss.backward()
    smoke_optimizer.step()
    print(f'batch {i} loss={loss.item():.4f}')
print('Smoke test seconds:', round(time.time() - t0, 2))


batch 0 loss=0.6558
batch 1 loss=0.7853
Smoke test seconds: 12.41


In [9]:
history = []
best_auc = -1.0
best_state = None
wait = 0

# Phase 1: train only KAN
freeze_backbone(model)
optimizer = torch.optim.Adam(model.kan.parameters(), lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

for epoch in range(1, PHASE1_EPOCHS + 1):
    tr = run_epoch(train_loader, optimizer=optimizer)
    va = run_epoch(val_loader, optimizer=None)
    scheduler.step(va['auc'] if not np.isnan(va['auc']) else 0.0)

    did_grid_update = False
    if GRID_UPDATE_ENABLED and epoch >= GRID_UPDATE_AFTER_EPOCH and epoch % GRID_UPDATE_EVERY == 0:
        did_grid_update = maybe_update_kan_grid(train_loader)

    row = {'phase': 1, 'epoch': epoch, 'grid_update': did_grid_update, **{f'tr_{k}': v for k, v in tr.items()}, **{f'va_{k}': v for k, v in va.items()}}
    history.append(row)
    print(f"P1 E{epoch:02d} | tr_auc={tr['auc']:.4f} va_auc={va['auc']:.4f} va_f1={va['f1_malignant']:.4f} grid_update={did_grid_update}")

    if va['auc'] > best_auc:
        best_auc = va['auc']
        wait = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_PATH)
    else:
        wait += 1
        if wait >= EARLY_STOP_PATIENCE:
            print('Early stop in phase 1')
            break

# Phase 2: unfreeze layer4 + KAN
model.load_state_dict(best_state)
unfreeze_layer4_and_kan(model)
optimizer = build_phase2_optimizer(model)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
wait = 0

for epoch in range(1, PHASE2_EPOCHS + 1):
    tr = run_epoch(train_loader, optimizer=optimizer)
    va = run_epoch(val_loader, optimizer=None)
    scheduler.step(va['auc'] if not np.isnan(va['auc']) else 0.0)

    did_grid_update = False
    if GRID_UPDATE_ENABLED and epoch >= GRID_UPDATE_AFTER_EPOCH and epoch % GRID_UPDATE_EVERY == 0:
        did_grid_update = maybe_update_kan_grid(train_loader)

    row = {'phase': 2, 'epoch': epoch, 'grid_update': did_grid_update, **{f'tr_{k}': v for k, v in tr.items()}, **{f'va_{k}': v for k, v in va.items()}}
    history.append(row)
    print(f"P2 E{epoch:02d} | tr_auc={tr['auc']:.4f} va_auc={va['auc']:.4f} va_f1={va['f1_malignant']:.4f} grid_update={did_grid_update}")

    if va['auc'] > best_auc:
        best_auc = va['auc']
        wait = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_PATH)
    else:
        wait += 1
        if wait >= EARLY_STOP_PATIENCE:
            print('Early stop in phase 2')
            break

model.load_state_dict(best_state)
print('Best val AUC:', round(best_auc, 4))
print('Saved:', BEST_PATH)


P1 E01 | tr_auc=0.5873 va_auc=0.6194 va_f1=0.5282 grid_update=True
P1 E02 | tr_auc=0.6623 va_auc=0.6211 va_f1=0.6358 grid_update=True
P1 E03 | tr_auc=0.6726 va_auc=0.6206 va_f1=0.2338 grid_update=True
P1 E04 | tr_auc=0.6928 va_auc=0.6189 va_f1=0.5237 grid_update=True
P1 E05 | tr_auc=0.6805 va_auc=0.6211 va_f1=0.4651 grid_update=True
P1 E06 | tr_auc=0.6801 va_auc=0.6196 va_f1=0.6060 grid_update=True
Early stop in phase 1
P2 E01 | tr_auc=0.6833 va_auc=0.6471 va_f1=0.5466 grid_update=True
P2 E02 | tr_auc=0.7232 va_auc=0.6507 va_f1=0.6509 grid_update=True
P2 E03 | tr_auc=0.7437 va_auc=0.6616 va_f1=0.6111 grid_update=True
P2 E04 | tr_auc=0.7524 va_auc=0.6678 va_f1=0.6491 grid_update=True
P2 E05 | tr_auc=0.7502 va_auc=0.6728 va_f1=0.6419 grid_update=True
P2 E06 | tr_auc=0.7678 va_auc=0.6855 va_f1=0.6417 grid_update=True
P2 E07 | tr_auc=0.7977 va_auc=0.6778 va_f1=0.6526 grid_update=True
P2 E08 | tr_auc=0.8038 va_auc=0.6717 va_f1=0.6490 grid_update=True
P2 E09 | tr_auc=0.7998 va_auc=0.6777 va_

In [ ]:
RUN_LBFGS_FINETUNE = True

In [10]:
# Etapa opcional: ajuste fino de la KAN con L-BFGS
if RUN_LBFGS_FINETUNE:
    freeze_backbone(model)
    lbfgs = torch.optim.LBFGS(model.kan.parameters(), lr=LBFGS_LR, max_iter=20, history_size=20)

    cache_batches = []
    for batch_idx, (x, y) in enumerate(train_loader):
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        cache_batches.append((x, y))
        if batch_idx + 1 >= LBFGS_MAX_BATCHES:
            break

    def closure():
        lbfgs.zero_grad()
        total = 0.0
        for x, y in cache_batches:
            logits = model(x)
            loss = criterion(logits, y)
            total = total + loss
        total.backward()
        return total

    for step in range(LBFGS_STEPS):
        loss = lbfgs.step(closure)
        print(f'LBFGS step {step + 1:02d} | loss={float(loss):.4f}')

    va = run_epoch(val_loader, optimizer=None)
    print('Val after LBFGS:', va)

    if va['auc'] > best_auc:
        best_auc = va['auc']
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_PATH)
        print('LBFGS improved the checkpoint.')
else:
    print('RUN_LBFGS_FINETUNE=False. Se omite esta etapa.')


RUN_LBFGS_FINETUNE=False. Se omite esta etapa.


In [11]:
hist_df = pd.DataFrame(history)
hist_df.to_csv(HISTORY_PATH, index=False)
display(hist_df.tail(10))
print('History saved:', HISTORY_PATH)


,phase,epoch,grid_update,tr_acc,tr_auc,tr_recall_malignant,tr_precision_malignant,tr_f1_malignant,tr_loss,va_acc,va_auc,va_recall_malignant,va_precision_malignant,va_f1_malignant,va_loss
6,2,1,True,0.630190,0.683304,0.678112,0.532435,0.596508,0.636344,0.601093,0.647129,0.530120,0.564103,0.546584,0.678233
7,2,2,True,0.663495,0.723152,0.726101,0.563803,0.634742,0.607206,0.597450,0.650663,0.827309,0.536458,0.650869,0.659443
8,2,3,True,0.669118,0.743700,0.767991,0.565665,0.651481,0.589743,0.617486,0.661640,0.662651,0.567010,0.611111,0.669199
9,2,4,True,0.676471,0.752364,0.757511,0.574919,0.653704,0.580980,0.635701,0.667771,0.742972,0.576324,0.649123,0.668418
10,2,5,True,0.669118,0.750183,0.756989,0.566372,0.647952,0.582247,0.613843,0.672751,0.763052,0.553936,0.641892,0.648938
11,2,6,True,0.677768,0.767833,0.773605,0.574502,0.659351,0.565996,0.633880,0.685549,0.722892,0.576923,0.641711,0.660832
12,2,7,True,0.701125,0.797706,0.775751,0.600000,0.676650,0.537111,0.641166,0.677798,0.742972,0.581761,0.652557,0.679713
13,2,8,True,0.715830,0.803829,0.785408,0.615643,0.690240,0.528281,0.637523,0.671653,0.738956,0.578616,0.649030,0.734388
14,2,9,True,0.716263,0.799789,0.801289,0.612983,0.694600,0.533835,0.642987,0.677677,0.690763,0.591065,0.637037,0.705329
15,2,10,True,0.735727,0.826435,0.824919,0.631579,0.715417,0.507692,0.652095,0.677102,0.730924,0.594771,0.655856,0.755590


History saved: G:\Cosas_programacion\Breast Cancer Interpretable-ml\reports\models\resnet18_kan_hybrid_gap32_history.csv


In [12]:
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
test_metrics = run_epoch(test_loader, optimizer=None)
print('Test metrics:', test_metrics)


Test metrics: {'acc': 0.5710900473933649, 'auc': 0.6472005932517612, 'recall_malignant': 0.7241379310344828, 'precision_malignant': 0.4864864864864865, 'f1_malignant': 0.581986143187067, 'loss': 0.6856345237713855}
